In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
!nvidia-smi

Fri May  8 07:49:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             28W /   70W |    5571MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [31]:
cd /content/drive/MyDrive/Net2Net_Up/

/content/drive/MyDrive/Net2Net_Up


In [32]:
import os
from threading import Thread  # for running the denoiser in parallel
import queue  # 队列
import numpy as np
import torch
import torch.optim
from models.skip import skip  # our network
from utils.utils import *  # auxiliary functions
from utils.data import Data  # class that holds img, psnr, time

from utils_drunet import utils_logger
from utils_drunet import utils_model
from utils_drunet import utils_image as util

# repalce ---> FFDNet ---> DRUNet
from models_drunet.network_unet import UNetRes as net

# from skimage.metrics import peak_signal_noise_ratio as compare_psnr1
# from skimage.metrics import structural_similarity as compare_ssim


import warnings
warnings.filterwarnings("ignore")

# got GPU? - if you are not getting the exact article results set CUDNN to False
CUDA_FLAG = True
CUDNN = True
if CUDA_FLAG:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    torch.backends.cudnn.enabled = CUDNN
    torch.backends.cudnn.benchmark = CUDNN
    dtype = torch.cuda.FloatTensor
else:
    dtype = torch.FloatTensor

X_LABELS = ['Iterations']*3
Y_LABELS = ['PSNR between x and net (db)', 'PSNR with original image (db)', 'loss']

ORIGINAL = 'Clean'
MASK='Mask'
CORRUPTED = 'Noisy'


In [33]:
# load image for inpainting (under noise) as mixed degradation (YUAN)
def load_image_inpainting_new(fclean_path,mask_path, fnoisy=True, sigma=25, plot=False):
    """
        fname - input file name
        d - Make dimensions divisible by `d`
        sigma - the amount of noise you want to add noise to the image
        Return a numpy image, and a noisy numpy image with sigma selected
    """
    img_pil, img_np           = load_and_crop_image(fclean_path)
    img_mask_pil, img_mask_np = load_and_crop_image(mask_path)


    # mask随着image.size调整大小
    img_mask_pil = img_mask_pil.resize(img_pil.size, Image.BICUBIC)
    img_mask_np = pil_to_np(img_mask_pil)

    # 图像要先乘mask:
    hx=img_np*(img_mask_np)

    # 然后再加噪(高斯噪声)：
    hx_n = np.clip(hx + np.random.normal(scale=sigma / 255 , size=hx.shape), 0, 1).astype(np.float32)

    # 然后再加噪(泊松噪声)：
    # hx_n = np.clip(hx + np.random.poisson(sigma/255, size=hx.shape), 0, 1).astype(np.float32) # lam>=0 值越小，噪声频率就越少

    data_dict = {ORIGINAL: Data(img_np), MASK:Data(img_mask_np),CORRUPTED: Data(hx_n, compare_psnr(img_np, hx_n))}

    if plot:
        plot_dict(data_dict)
    return data_dict

In [34]:
# 常规图像测试部分：
SIGMA = 3 # small:10, medium:25, heavy:50
# 图像的路径

img_name = '084'

img_path  = '/content/drive/MyDrive/Net2Net_Up/0_data/Dunhuang_data/%s.jpg'%img_name
mask_path = '/content/drive/MyDrive/Net2Net_Up/0_data/Dunhuang_data/%s_mask.jpg'%img_name
data_dict = load_image_inpainting_new(img_path,mask_path, sigma=SIGMA, plot=True)

Output hidden; open in https://colab.research.google.com to view.

In [35]:
# y=data_dict[CORRUPTED].img
# plt.imsave('./INPUT.png',y.transpose(1,2,0))

In [36]:
def get_network_and_input(img_shape, input_depth=32, pad='reflection',
                          upsample_mode='bilinear', use_interpolate=True, align_corners=False,
                          act_fun='LeakyReLU', skip_n33d=128, skip_n33u=128, skip_n11=4,
                          num_scales=5, downsample_mode='stride', INPUT='noise'):  # 'meshgrid'
    """ Getting the relevant network and network input (based on the image shape and input depth)
        We are using the same default params as in DIP article
        img_shape - the image shape (ch, x, y)
    """
    n_channels = img_shape[0]
    net = skip(input_depth, n_channels,
               num_channels_down=[skip_n33d] * num_scales if isinstance(skip_n33d, int) else skip_n33d,
               num_channels_up=[skip_n33u] * num_scales if isinstance(skip_n33u, int) else skip_n33u,
               num_channels_skip=[skip_n11] * num_scales if isinstance(skip_n11, int) else skip_n11,
               upsample_mode=upsample_mode, use_interpolate=use_interpolate, align_corners=align_corners,
               downsample_mode=downsample_mode, need_sigmoid=True, need_bias=True, pad=pad, act_fun=act_fun).type(dtype)
    net_input = get_noise(input_depth, INPUT, img_shape[1:]).type(dtype).detach()
    return net, net_input

In [37]:
# ACM-MM-ASIA使用的深度去噪器DRUNet:
def DRUNet_rgb_yuan(model, noisy_np_img,sigma):

    for k, v in model.named_parameters():
        v.requires_grad = False

    model = model.to(device)


    # 预处理：
    noisy_np_img = np.transpose(noisy_np_img,(1,2,0))
    # print(noisy_np_img.shape)
    img_L = util.single2tensor4(noisy_np_img)
    img_L = torch.cat((img_L, torch.FloatTensor([sigma/255.]).repeat(1, 1, img_L.shape[2], img_L.shape[3])), dim=1)
    img_L = img_L.to(device)
    # print(img_L.shape) # torch.Size([1, 2, 256, 256])



    img_E = utils_model.test_mode(model, img_L, refield=64, mode=5)  # 执行
    img_E = img_E.cpu()

    return np.array(img_E, dtype=np.float32)

In [38]:
import numpy as np
import cv2
from skimage.metrics import structural_similarity as ssim

def calculate_y_channel_ssim(img_A, img_B):
    """
    计算两张图片在 YUV 颜色空间下的 Y 通道 SSIM 值。

    参数:
    img_A: numpy.ndarray, 形状为 (3, height, width)
    img_B: numpy.ndarray, 形状为 (3, height, width)

    返回:
    SSIM_Y: float, Y 通道的 SSIM 值
    """
    # 调整通道顺序，从 (3, H, W) 变为 (H, W, 3)
    img_A = np.transpose(img_A, (1, 2, 0))
    img_B = np.transpose(img_B, (1, 2, 0))

    # 转换为 YUV 颜色空间
    img_A_YUV = cv2.cvtColor(img_A, cv2.COLOR_RGB2YUV)
    img_B_YUV = cv2.cvtColor(img_B, cv2.COLOR_RGB2YUV)

    # 提取 Y 通道
    Y_A = img_A_YUV[:, :, 0]
    Y_B = img_B_YUV[:, :, 0]

    # 计算 SSIM
    SSIM_Y = ssim(Y_A, Y_B, data_range=Y_A.max() - Y_A.min())

    return SSIM_Y


In [39]:
def train_via_admm(net, net_input, denoiser_function, y, mask, HR_img,  # D is the downsampler, y is LR image
                   algorithm_name="", save_path="",          # will save params and graphs in this folder
                   admm_iter=100, LR=0.001, update_iter=10, method='fixed_point',   # 'fixed_point' or 'grad' or 'mixed'  LR: 0.008--> 0.001
                   # sigma_f:FFDNET先验的输入参数
                   sigma_f=SIGMA, beta=0.01, mu=0.2, LR_x=None, noise_factor=0.043):    # LR_x needed only if method!=fixed_point 0.033


    # get optimizer and loss function:
    optimizer = torch.optim.Adam(net.parameters(), lr=LR)  # using ADAM opt

    mse = torch.nn.MSELoss().type(dtype)  # using MSE loss

    # additional noise added to the input:
    net_input_saved = net_input.detach().clone()
    noise = net_input.detach().clone()

    # x update method:
    if method == 'fixed_point':
        swap_iter = admm_iter + 1
        LR_x = None
    elif method == 'grad':
        swap_iter = -1
    elif method == 'mixed':
        swap_iter = admm_iter // 2
    else:
        assert False, "method can only be 'fixed_point' or 'grad' or 'mixed' !"

    # initialize:
    x = np.zeros_like(y)  # bicubic.copy()

    y_torch = np_to_torch(y).type(dtype) # y 就是DIP进行超分的目标(y_torch是其tensor)
    mask_torch = np_to_torch(mask).type(dtype)

    f_x = x.copy()
    avg = np.rint(y)

    # 拉格朗日乘子U初始化：
    u = np.zeros_like(x)

    psnr_avg_list=[]
    ssim_avg_list=[]
    image_list  = []

    psnr_avg_list=[]
    image_list=[]

    for i in range(1, 1 + admm_iter):

        # step 1, update network, eq. 7 in the article
        optimizer.zero_grad()
        net_input = net_input_saved + (noise.normal_() * noise_factor)
        out = net(net_input)
        out_np = torch_to_np(out)

        # loss:
        loss_y = mse(out*mask_torch, y_torch)

        loss_x = mse(out, np_to_torch(x - u).type(dtype))
        total_loss = loss_y + mu * loss_x  # SR任务上总的loss
        total_loss.backward()
        optimizer.step()

        # step 2, update x using a denoiser and result from step 1
        if i % update_iter == 0:
            f_x = denoiser_function(model, x.copy(), sigma_f)

        # 因为随着迭代噪声变小，减少sigma_f的影响 - 0.000001
        sigma_f = sigma_f

        # 使用深度先验的话需要去掉一维：
        f_x=np.squeeze(f_x)

        # step 2, update x using a the denoiser (f_x) and network outputs (out_np)
        if i < swap_iter:
            x = 1 / (beta + mu) * (beta * f_x + mu * (out_np + u))      # eq. 11 in the article
        else:
            x = x - LR_x * (beta * (x - f_x) + mu * (x - out_np - u))   # eq. 12 in the article
        np.clip(x, 0, 1, out=x)  # making sure that image is in bounds

        # step 3, update u
        u = u + out_np - x

        avg = avg * .99 + out_np * .01


        psnr_avg = compare_PSNR(HR_img, avg,  on_y=True)  # HR_img/avg.shape:[3,m,n],二者都是numpy的格式

        ssim_avg = calculate_y_channel_ssim(HR_img, avg)

        psnr_avg_list.append(psnr_avg)
        image_list.append(avg)

        psnr_avg_max_temp=max(psnr_avg_list)

        psnr_avg_max_temp_index=psnr_avg_list.index(psnr_avg_max_temp)

        ssim_avg_list.append(ssim_avg)
        # ssim_avg_max_temp=max(ssim_avg_list)

        # ssim_avg_max_temp_index=ssim_avg_list.index(ssim_avg_max_temp)

        print('\r', '%04d/%04d Loss %f' % (i, admm_iter, total_loss.item()),' 目前最佳PSNR_avg: %.5f 目前最佳PSNR_avg的迭代数: %d '
              % (psnr_avg_max_temp,psnr_avg_max_temp_index), end='')


    return psnr_avg_list,ssim_avg_list,image_list


In [ ]:
def run_and_plot(denoiser, name):
    global data_dict
    net, net_input = get_network_and_input(img_shape=data_dict[ORIGINAL].img.shape)

    psnr_avg_list,ssim_avg_list,image_list = train_via_admm(net, net_input, denoiser, y=data_dict[CORRUPTED].img,
                                                              mask=data_dict[MASK].img,
                                                              HR_img=data_dict[ORIGINAL].img,
                                                              admm_iter=3000,
                                                              algorithm_name=name,
                                                              sigma_f=SIGMA)

    return psnr_avg_list,ssim_avg_list,image_list

# import time
# T1 = time.time()

n_channels=3  # 处理3通道图像
model_path ='/content/drive/MyDrive/Net2Net_Up/model_zoo/drunet_color.pth'
# model_path ='/home/yuanweimin/PHD_3/YUAN_LASTEST_WORK_对比算法/2022_DPIR_TPAMI/model_zoo/drunet_color.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = net(in_nc=n_channels+1, out_nc=n_channels, nc=[64, 128, 256, 512], nb=4, act_mode='R', downsample_mode="strideconv", upsample_mode="convtranspose")
model.load_state_dict(torch.load(model_path), strict=True)
model.eval()

psnr_avg_list,ssim_avg_list,image_list = run_and_plot(DRUNet_rgb_yuan, 'fixed_point')

# T2 = time.time()
# print('程序运行时间:%s秒' % ((T2 - T1)/1000))


 0038/3000 Loss 0.011617  目前最佳PSNR_avg: 12.93932 目前最佳PSNR_avg的迭代数: 37 

In [ ]:
# 获取psnr_avg_list列表的最大值和对应索引
net_max=max(psnr_avg_list)
net_max_index=psnr_avg_list.index(net_max)

ssim_max = ssim_avg_list[net_max_index]

optimal_img=image_list[net_max_index]

print('max iteration at: ',net_max_index, 'max_psnr is: ',net_max, 'max_ssim is: ', ssim_max)

In [ ]:
# part2: 显示最优的那张结果：
# optimal_img=image_list[psnr_avg_max_temp_index]


# import matplotlib.pyplot as plt
np.clip(optimal_img, 0, 1, out=optimal_img)

# save result
plt.imsave('/content/drive/MyDrive/Net2Net_Up/%s.png'%img_name,(optimal_img).transpose(1,2,0))